# PROYECTO PTIA 2026-2
## StudyFlow AI: capa de asignación

**Integrantes:**

Karol Ximena Rodriguez Reyes
Juan David Moreno D'Aleman

---

### ¿Qué vamos a hacer en este notebook?

`LLM (entiende lo que pide el estudiante)` -> `Validador (¿cabe en el tiempo?)` -> `MLP (prioriza y reparte)` -> **`Capa de asignación (arma las sesiones)`** -> `Validador de salida + LLM redactor`

El MLP solo entrega **dos resultados por actividad**: una prioridad (0 a 1) y un perfil de reparto en 3 tramos. No sabe de días, horas ni franjas, así que todavía no hay horario. Aquí desarrollamos la capa **determinista** que convierte esos resultados en sesiones concretas `(actividad, día, hora inicio, hora fin)` y garantiza que ninguna sesión viole la disponibilidad ni la fecha límite.

Se prueba aquí antes de pasarla a archivos `.py`. Flujo: `plan validado + salidas del MLP -> asignar -> sesiones + resumen por actividad -> front y LLM redactor`.

Todo el código, a excepción de las pruebas, se mudará a archivos `.py` para conectar los módulos entre sí y el front con el back.

# PASO 0. DEPENDENCIAS

Las tres celdas siguientes son copia **exactas** de `models.py`, `disponibilidad.py` y `validador.py`, tal como estan en `StudyFlow_CapaValidacion.ipynb`. Están aquí solo para que este notebook corra solo; al migrar se reemplazan por `import`.

In [1]:
"""Modelos de entrada (forma y valores). Ver StudyFlow_CapaValidacion.ipynb, paso 2."""
from datetime import date, time
from enum import Enum, IntEnum

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


class Nivel(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class DuracionSesion(IntEnum):
    M30 = 30
    M60 = 60
    M90 = 90
    M120 = 120
    M150 = 150
    M180 = 180
    M210 = 210
    M240 = 240


class Actividad(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    nombre: str = Field(min_length=1, max_length=100)
    prioridad: Nivel
    dificultad: Nivel
    fecha_limite: date
    horas_estimadas: float = Field(gt=0, le=100)

    @field_validator("prioridad", "dificultad", mode="before")
    @classmethod
    def _nivel_sin_mayusculas(cls, v):
        return v.strip().lower() if isinstance(v, str) else v


class Franja(BaseModel):
    inicio: time
    fin: time

    @model_validator(mode="after")
    def _fin_despues_de_inicio(self):
        if self.fin <= self.inicio:
            raise ValueError("la franja debe terminar después de empezar")
        return self


class DisponibilidadDia(BaseModel):
    dia: date
    franjas: list[Franja]

    @model_validator(mode="after")
    def _sin_traslapes(self):
        ordenadas = sorted(self.franjas, key=lambda f: f.inicio)
        for a, b in zip(ordenadas, ordenadas[1:]):
            if b.inicio < a.fin:
                raise ValueError(f"las franjas {a.inicio}-{a.fin} y {b.inicio}-{b.fin} se traslapan")
        return self


class PlanEntrada(BaseModel):
    actividades: list[Actividad] = Field(min_length=1)
    disponibilidad: list[DisponibilidadDia] = Field(min_length=1)
    max_horas_dia: float | None = Field(default=None, gt=0, le=8)
    duracion_sesion: DuracionSesion

    @model_validator(mode="after")
    def _sin_dias_repetidos(self):
        dias = [d.dia for d in self.disponibilidad]
        if len(dias) != len(set(dias)):
            raise ValueError("hay días repetidos en la disponibilidad")
        return self

In [2]:
"""Capacidad efectiva de estudio por día. Ver StudyFlow_CapaValidacion.ipynb, paso 3."""
from datetime import date, time

BLOQUE_MIN = 30  # unidad de capacidad y duración mínima de una sesión (minutos)


def _minutos(t: time):
    return t.hour * 60 + t.minute


def capacidad_dia(dia: DisponibilidadDia, max_horas_dia: float | None = None):
    minutos = sum((_minutos(f.fin) - _minutos(f.inicio)) // BLOQUE_MIN * BLOQUE_MIN for f in dia.franjas)
    if max_horas_dia is not None:
        minutos = min(minutos, int(max_horas_dia * 60) // BLOQUE_MIN * BLOQUE_MIN)
    return minutos / 60


def capacidad_entre(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    """Horas efectivas de los días d con inicio <= d < fin (el día `fin` no cuenta)."""
    return sum(capacidad_dia(d, max_horas_dia) for d in disponibilidad if inicio <= d.dia < fin)


def dias_con_capacidad(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    return sum(1 for d in disponibilidad if inicio <= d.dia < fin and capacidad_dia(d, max_horas_dia) > 0)

In [3]:
"""Validación de coherencia del plan (R1-R7). Ver StudyFlow_CapaValidacion.ipynb, paso 4."""
from collections import Counter
from datetime import date
from enum import Enum
from typing import Any, Literal

from pydantic import BaseModel, Field

MARGEN_HORAS = 0.0
VENTANA_MINIMA_DIAS = 3
_ORDEN_PRIORIDAD = {Nivel.BAJA: 0, Nivel.MEDIA: 1, Nivel.ALTA: 2}


class Estado(str, Enum):
    FACTIBLE = "factible"
    CON_ADVERTENCIAS = "factible_con_advertencias"
    INVIABLE = "inviable"


class Hallazgo(BaseModel):
    codigo: str
    mensaje: str
    datos: dict[str, Any] = Field(default_factory=dict)


class Resultado(BaseModel):
    estado: Estado
    errores: list[Hallazgo]
    advertencias: list[Hallazgo]


def _sugerir_recorte(vencen, deficit):
    candidatas = sorted(vencen, key=lambda a: (_ORDEN_PRIORIDAD[a.prioridad], -a.horas_estimadas))
    elegidas, recortado = [], 0.0
    for a in candidatas:
        if recortado >= deficit:
            break
        elegidas.append({"nombre": a.nombre, "horas": a.horas_estimadas})
        recortado += a.horas_estimadas
    return elegidas


def validar(plan: PlanEntrada, fecha_inicio: date, modo: Literal["nuevo", "replanificacion"] = "nuevo") -> Resultado:
    errores: list[Hallazgo] = []
    advertencias: list[Hallazgo] = []
    disp, tope = plan.disponibilidad, plan.max_horas_dia
    vigentes = [a for a in plan.actividades if a.fecha_limite >= fecha_inicio]

    if tope is not None and plan.duracion_sesion / 60 > tope:  # R3a
        errores.append(Hallazgo(
            codigo="R3A_SESION_MAYOR_MAXIMO",
            mensaje=f"La sesión máxima ({int(plan.duracion_sesion)} min) supera el máximo diario ({tope:g} h)",
            datos={"duracion_sesion_min": int(plan.duracion_sesion), "max_horas_dia": tope}))

    for a in plan.actividades:  # R4
        if a.fecha_limite < fecha_inicio:
            errores.append(Hallazgo(
                codigo="R4_FECHA_PASADA",
                mensaje=f"'{a.nombre}' vence el {a.fecha_limite}, antes del inicio del plan ({fecha_inicio})",
                datos={"actividad": a.nombre, "fecha_limite": a.fecha_limite.isoformat()}))

    pasados = sorted(d.dia for d in disp if d.dia < fecha_inicio)  # R5
    if pasados:
        advertencias.append(Hallazgo(
            codigo="R5_DISPONIBILIDAD_PASADA",
            mensaje=f"Se ignoran {len(pasados)} día(s) de disponibilidad anteriores al inicio del plan",
            datos={"dias": [d.isoformat() for d in pasados]}))

    repetidos = sorted(n for n, c in Counter(a.nombre.casefold() for a in plan.actividades).items() if c > 1)  # R6
    if repetidos:
        advertencias.append(Hallazgo(
            codigo="R6_NOMBRE_DUPLICADO",
            mensaje=f"Hay actividades con el mismo nombre: {', '.join(repetidos)}",
            datos={"nombres": repetidos}))

    for d in sorted(disp, key=lambda d: d.dia):  # R3b
        if d.dia >= fecha_inicio and d.franjas and capacidad_dia(d) == 0:
            advertencias.append(Hallazgo(
                codigo="R3B_DIA_SIN_CAPACIDAD",
                mensaje=f"El {d.dia} no aporta horas: todas sus franjas duran menos de 30 min",
                datos={"dia": d.dia.isoformat()}))

    for a in vigentes:  # R2
        n = dias_con_capacidad(disp, tope, fecha_inicio, a.fecha_limite)
        if n < VENTANA_MINIMA_DIAS:
            hallazgo = Hallazgo(
                codigo="R2_VENTANA_MINIMA",
                mensaje=f"'{a.nombre}' tiene {n} día(s) con disponibilidad antes de la entrega; el mínimo es {VENTANA_MINIMA_DIAS}",
                datos={"actividad": a.nombre, "dias_con_disponibilidad": n, "minimo": VENTANA_MINIMA_DIAS})
            (errores if modo == "nuevo" else advertencias).append(hallazgo)

    for limite in sorted({a.fecha_limite for a in vigentes}):  # R1 (cubre R7)
        vencen = [a for a in vigentes if a.fecha_limite <= limite]
        carga = sum(a.horas_estimadas for a in vencen)
        capacidad = capacidad_entre(disp, tope, fecha_inicio, limite)
        deficit = carga - (capacidad + MARGEN_HORAS)
        if deficit > 1e-9:
            errores.append(Hallazgo(
                codigo="R1_CAPACIDAD_INSUFICIENTE",
                mensaje=f"Hasta el {limite} hay {carga:g} h de trabajo y solo {capacidad:g} h disponibles (faltan {deficit:g} h)",
                datos={"fecha_critica": limite.isoformat(), "horas_requeridas": carga, "horas_disponibles": capacidad,
                       "deficit_horas": round(deficit, 4), "sugerencia_recorte": _sugerir_recorte(vencen, deficit)}))

    estado = Estado.INVIABLE if errores else Estado.CON_ADVERTENCIAS if advertencias else Estado.FACTIBLE
    return Resultado(estado=estado, errores=errores, advertencias=advertencias)

# PASO 1. REGLAS Y DECISIONES

## 1.1 Entradas y salidas

| | Qué es |
|---|---|
| Entrada | `PlanEntrada` ya validado, `fecha_inicio`, una `Prediccion` por actividad y, opcional, `ocupadas` (sesiones ya cumplidas o fijadas) |
| Salida | `ResultadoAsignacion`: la lista de `Sesion` y un resumen por actividad (`asignada`, `parcial` o `no_asignada`, con horas asignadas y faltantes) |

Las predicciones se alinean con las actividades **por posición**, no por nombre: el validador solo advierte (R6) cuando hay nombres repetidos.

## 1.2 Cómo trabaja el algoritmo

Se procesa una actividad a la vez, de mayor a menor prioridad, y cada una marca como ocupados los bloques que toma.

1. **Orden:** prioridad del MLP de mayor a menor; a igualdad, fecha límite más cercana y luego nombre (el resultado es reproducible). La más prioritaria escoge primero.
2. **Bloques totales:** `horas_estimadas` entre 30 min, redondeado al bloque más cercano (mínimo 1).
3. **Sesión única:** si la prioridad es **media o baja** y las horas caben en una sesión (`bloques <= duracion_sesion / 30`), la actividad se ubica **completa en un solo hueco seguido** y no se reparte. Se escoge el tramo que más pesa en su perfil (si ahí no hay hueco, el tramo vecino más cercano; a igual distancia, el posterior); dentro del tramo, el día menos cargado; a igual carga, el hueco que deja un resto par (así no queda media hora suelta) y, después, el más ajustado. Si ningún día tiene un hueco seguido de ese tamaño, la actividad sigue por el camino normal (pasos 5 a 10) y queda partida. La prioridad **alta** no usa esta regla: se reparte para adelantar el trabajo.
4. **Ventana:** los días desde `fecha_inicio` hasta antes de la fecha límite que **todavía tengan bloques libres** (después de lo que tomaron las actividades anteriores). Se parten en 3 tramos consecutivos y el sobrante va a los primeros: 7 días dan 3, 2 y 2.
5. **Piezas de 1 h:** para que no queden sesiones sueltas de 30 min, el trabajo se reparte en **piezas** de `SESION_MIN` = 60 min (2 bloques seguidos). Los bloques totales se dividen en piezas enteras y un resto de a lo sumo 1 bloque; por ejemplo 2 h 30 son 2 piezas y 1 bloque. Si `duracion_sesion` es de 30 min, la pieza es de 1 bloque (el máximo manda sobre el mínimo).
6. **Piezas por tramo:** `perfil × piezas`, redondeado con el **método del mayor residuo**: cada tramo recibe su parte entera y las piezas que faltan se dan a los tramos con mayor parte decimal. La suma queda exacta.

   Ejemplo: 8 h = 8 piezas, perfil `[0.45, 0.35, 0.20]`. Lo ideal es 3.6, 2.8 y 1.6 piezas; con partes enteras 3, 2 y 1 sobran 2, que van a los tramos con residuo 0.8 y 0.6 (el más temprano en empate). Resultado: 4, 3 y 1 piezas = **4 h, 3 h y 1 h**.
7. **Capacidad del tramo:** una pieza cabe donde haya **2 bloques libres seguidos**; un hueco de 30 min, o el bloque que sobra de una franja de número impar de bloques, no sirve para una pieza. Si un tramo no tiene espacio para sus piezas, se queda con las que caben y el excedente pasa al **tramo vecino más cercano con espacio**; a igual distancia, al **posterior**. Un tramo sin días (ventana corta) entrega todo a sus vecinos.
8. **Reparto en días:** dentro de un tramo, pieza a pieza al día con menos carga de esta actividad que aún tenga espacio (empate: el más temprano). Es un reparto parejo que respeta lo que ya ocuparon las actividades de mayor prioridad. En cada día se toman las piezas libres más tempranas.
9. **Resto y lo que no cupo en piezas:** la media hora que sobra se pega a un bloque ya tomado de la actividad (el día menos cargado), así una sesión de 1 h pasa a 1 h 30. Si no hay dónde pegarla, va a cualquier hueco libre de la ventana.
10. **Limpieza de sesiones cortas:** cuando todas las actividades están ubicadas, se revisa cada hueco del calendario. Si una actividad (que no sea de sesión única) quedó con un grupo que da una sesión de menos de 1 h, se prueba, en este orden: traer ahí un bloque de otra sesión de la misma actividad, llevar el bloque a otro hueco donde la actividad ya estudia (directo, o cambiándolo por un bloque de otra actividad que en los dos huecos quede en sesiones de al menos 1 h). Nunca cambia cuántas horas tiene cada actividad ni cuántas se estudian cada día.
11. **Sesiones:** los bloques consecutivos de cada día se unen en sesiones de máximo `duracion_sesion`. Si una tanda pasa del máximo, se parte en sesiones parejas (7 bloques con máximo de 3 h dan 2 h y 1 h 30, no 3 h y 30 min). Una sesión nunca cruza el hueco entre dos franjas.
12. **Dos órdenes:** como las actividades de sesión única necesitan un hueco seguido y las demás lo fragmentan, se arma el horario dos veces: por prioridad, y con las de sesión única primero. Se queda el de menor **costo**, comparado en este orden: horas sin ubicar, sesiones de menos de 1 h (salvo en actividades de media hora) y actividades de sesión única que quedaron partidas. A igual costo gana el orden por prioridad.
13. **Respaldo:** si todavía quedan horas sin ubicar, se prueba además sin las reglas de sesión única y de limpieza y con bloques de 30 min, y se queda lo que deje **menos horas sin ubicar**. Así pedir sesiones de al menos 1 h o de una sola vez nunca cuesta horas asignadas.
14. **Lo que no cabe** se reporta, nunca se fuerza: `parcial` (se asigna lo que cabe y se informan las horas faltantes) o `no_asignada`.

## 1.3 Bloques libres de un día

Es el mismo cálculo de capacidad del validador: cada franja se redondea hacia abajo a bloques de 30 min contados desde su inicio y se respeta `max_horas_dia`. Para replanificar, se descuentan los bloques que toca una sesión de `ocupadas`, y esas horas **también cuentan para el máximo diario**.

## 1.4 Decisiones

* "Hacia adelante" en el excedente significa **hacia tramos posteriores** del calendario; nunca se pasa de la fecha límite porque la ventana termina antes de ella.
* Horas que no son múltiplo de 30 min (por ejemplo 2.2 h) se redondean al bloque más cercano; `horas_solicitadas` conserva el valor original y `horas_asignadas` lo que se ubicó.
* **Sesión mínima de 1 h:** se prefirió a un reparto en bloques de 30 min porque una sesión de media hora rinde poco y llena el calendario de piezas sueltas. Quedan sesiones de 30 min solo cuando la actividad es de media hora, cuando la disponibilidad es tan justa que no hay otra forma de ubicar todo (por ejemplo, 34 h en 34,5 h libres), o cuando sale del respaldo del paso 13.
* **Sesión única para prioridad media y baja:** una actividad que cabe en una sesión no se parte en días. Es una regla de comodidad (menos cambios de contexto para algo poco urgente) y se cede ante lo que falta ubicar: si no hay hueco seguido, o si ubicar todo exige partirla, se parte antes que dejar horas sin ubicar.
* **Efecto en el reparto:** con piezas de 1 h el reparto parejo es más grueso (un día recibe 2 h o 1 h, ya no 1 h 30 contra 1 h) y el día más cargado sube un poco.
* La capa **no vuelve a validar** el plan (asume que pasó por el validador), pero nunca ubica una sesión fuera de las franjas, antes de `fecha_inicio` ni desde el día de la fecha límite. El paso 5 lo comprueba en cada caso.
* La ventana se calcula con los días que tienen espacio **en ese momento**: una actividad de menor prioridad reparte su trabajo entre lo que dejaron las anteriores.

## PASO 2. MODELOS DE LA CAPA (`asignacion.py`, primera parte)

| Modelo | Reglas / uso |
|---|---|
| `Prediccion` | salida del MLP para una actividad: `prioridad` entre 0 y 1; `perfil` de 3 proporciones no negativas que suman 1 (tolerancia 0.001, por el redondeo del softmax) |
| `Sesion` | `actividad`, `dia`, `inicio`, `fin`. Es la salida y también la entrada `ocupadas` |
| `EstadoAsignacion` | `asignada`, `parcial`, `no_asignada` |
| `ResumenActividad` | nombre, prioridad, horas solicitadas, asignadas y faltantes, estado |
| `ResultadoAsignacion` | sesiones (ordenadas por día y hora) y resumen por actividad (en orden de prioridad) |

In [4]:
"""Modelos de la capa de asignación. Ver StudyFlow_CapaAsignacion.ipynb, paso 2."""
from datetime import date, time
from enum import Enum

from pydantic import BaseModel, Field, field_validator

TRAMOS = 3
SESION_MIN = 60  # minutos: toda sesión dura al menos esto, salvo que no quede otra (media hora suelta, hueco de 30 min)


class Prediccion(BaseModel):
    prioridad: float = Field(ge=0, le=1)
    perfil: list[float] = Field(min_length=TRAMOS, max_length=TRAMOS)

    @field_validator("perfil")
    @classmethod
    def _perfil_valido(cls, v):
        if min(v) < 0 or abs(sum(v) - 1) > 1e-3:
            raise ValueError("el perfil debe tener proporciones >= 0 que sumen 1")
        return v


class Sesion(BaseModel):
    actividad: str
    dia: date
    inicio: time
    fin: time


class EstadoAsignacion(str, Enum):
    ASIGNADA = "asignada"
    PARCIAL = "parcial"
    NO_ASIGNADA = "no_asignada"


class ResumenActividad(BaseModel):
    nombre: str
    prioridad: float
    horas_solicitadas: float
    horas_asignadas: float
    horas_faltantes: float
    estado: EstadoAsignacion


class ResultadoAsignacion(BaseModel):
    sesiones: list[Sesion]
    actividades: list[ResumenActividad]

## 2.1 Pruebas de los modelos

In [5]:
from pydantic import ValidationError


def rechaza(fn, fragmento):
    try:
        fn()
    except ValidationError as e:
        assert fragmento in str(e), str(e)
        return
    raise AssertionError(f"debía rechazar ({fragmento})")


franjas = lambda *pares: [{"inicio": i, "fin": f} for i, f in pares]


def dia(fecha, *pares):
    return DisponibilidadDia(dia=fecha, franjas=franjas(*pares))


p = Prediccion(prioridad=0.87, perfil=[0.45, 0.35, 0.20])
assert p.prioridad == 0.87
Prediccion(prioridad=0, perfil=[1, 0, 0])
Prediccion(prioridad=1, perfil=[0.333333, 0.333333, 0.333334])
Prediccion(prioridad=0.5, perfil=[0.2999999, 0.7, 0.0000001])      # softmax de Keras: suma 1 con error ínfimo
rechaza(lambda: Prediccion(prioridad=1.2, perfil=[0.5, 0.3, 0.2]), "less_than_equal")
rechaza(lambda: Prediccion(prioridad=-0.1, perfil=[0.5, 0.3, 0.2]), "greater_than_equal")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.5, 0.5]), "too_short")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.4, 0.3, 0.2, 0.1]), "too_long")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.5, 0.5, 0.5]), "sumen 1")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[1.2, -0.2, 0.0]), "sumen 1")

s = Sesion(actividad="Taller", dia="2026-10-26", inicio="18:00", fin="19:00")
assert s.dia == date(2026, 10, 26) and s.fin.hour == 19

print("Modelos de la asignación: todas las pruebas pasaron")

Modelos de la asignación: todas las pruebas pasaron


# PASO 3. ALGORITMO DE ASIGNACIÓN (`asignacion.py`, segunda parte)

Cada paso del apartado 1.2 es una función pequeña, para probarlas por separado y para que `asignar` se lea de arriba abajo:

| Función | Paso | Qué hace |
|---|---|---|
| `_bloques_dia` | 1.3 | bloques libres de un día (franja más temprana primero) |
| `_ordenar` | 1 | orden de las actividades |
| `_dividir_dias` | 4 | parte la ventana en 3 tramos |
| `_corridas` | 3 y 10 | grupos de bloques seguidos (un hueco del calendario) |
| `_ubicar_unica` | 3 | hueco seguido para una actividad de sesión única |
| `_parejas` | 5 y 7 | grupos de bloques seguidos (las piezas de 1 h) que caben en un día |
| `_repartir_tramos` | 6 | piezas por tramo (mayor residuo) |
| `_ajustar_a_capacidad` | 7 | recorta a lo que cabe y traslada el excedente |
| `_repartir_en_dias` | 8 | reparto parejo entre los días de un tramo |
| `_completar` | 9 | ubica el resto de 30 min y lo que no cupo en piezas |
| `_valida` y `_limpiar_cortas` | 10 | quitan las sesiones de menos de 1 h moviendo bloques entre huecos |
| `_a_sesiones` | 11 | une bloques consecutivos en sesiones |
| `_asignar` | 3 a 11 y 14 | el algoritmo completo con un tamaño de pieza y un orden dados |
| `_costo` | 12 | mide qué tan bueno es un horario: horas sin ubicar, sesiones cortas, sesiones únicas partidas |
| `asignar` | 12 y 13 | función que usan los demás módulos: prueba los dos órdenes y, si algo no cabe, el respaldo |


In [6]:
"""Capa de asignación determinista. Ver StudyFlow_CapaAsignacion.ipynb, paso 3."""
from datetime import date, time
from math import ceil, floor


def _hora(minutos: int):
    return time(minutos // 60, minutos % 60)


def _bloques_dia(dia: DisponibilidadDia, tope: float | None, ocupadas: list[Sesion]):
    """Inicio (minutos del día) de los bloques libres, del más temprano al más tardío."""
    bloques = []
    for f in sorted(dia.franjas, key=lambda f: f.inicio):
        ini, fin = _minutos(f.inicio), _minutos(f.fin)
        bloques += range(ini, ini + (fin - ini) // BLOQUE_MIN * BLOQUE_MIN, BLOQUE_MIN)
    libres = [b for b in bloques
              if not any(_minutos(s.inicio) < b + BLOQUE_MIN and b < _minutos(s.fin) for s in ocupadas)]
    if tope is not None:
        usados = sum((_minutos(s.fin) - _minutos(s.inicio)) // BLOQUE_MIN for s in ocupadas)
        libres = libres[:max(int(tope * 60) // BLOQUE_MIN - usados, 0)]
    return libres


def _ordenar(actividades: list[Actividad], predicciones: list[Prediccion]):
    """Índices de las actividades de mayor a menor prioridad (empate: fecha límite y nombre)."""
    return sorted(range(len(actividades)),
                  key=lambda i: (-predicciones[i].prioridad, actividades[i].fecha_limite, actividades[i].nombre))


def _repartir_tramos(total: int, perfil: list[float]) -> list[int]:
    """Bloques por tramo con el método del mayor residuo; la suma es exactamente `total`."""
    suma = sum(perfil)
    ideal = [p / suma * total for p in perfil]
    base = [floor(x + 1e-9) for x in ideal]
    por_dar = sorted(range(TRAMOS), key=lambda i: (-(ideal[i] - base[i]), i))
    for i in por_dar[:total - sum(base)]:
        base[i] += 1
    return base


def _dividir_dias(dias: list[date]):
    """Parte los días en 3 tramos consecutivos; los primeros reciben el sobrante."""
    n, resto = divmod(len(dias), TRAMOS)
    tramos, i = [], 0
    for t in range(TRAMOS):
        k = n + (t < resto)
        tramos.append(dias[i:i + k])
        i += k
    return tramos


def _ajustar_a_capacidad(quieren: list[int], caben: list[int]):
    """Recorta cada tramo a su capacidad y pasa el excedente al vecino más cercano con espacio
    (a igual distancia, el posterior). Devuelve (bloques por tramo, bloques sin ubicar)."""
    puestos = [min(q, c) for q, c in zip(quieren, caben)]
    sobras = [q - p for q, p in zip(quieren, puestos)]
    sin_ubicar = 0
    for i, sobra in enumerate(sobras):
        for j in sorted(range(TRAMOS), key=lambda j: (abs(j - i), -j)):
            dar = min(sobra, caben[j] - puestos[j])
            puestos[j] += dar
            sobra -= dar
        sin_ubicar += sobra
    return puestos, sin_ubicar


def _repartir_en_dias(cantidad: int, libres: dict[date, int]):
    """Reparte `cantidad` bloques parejo entre los días, sin pasar el espacio libre de cada uno."""
    puestos = dict.fromkeys(libres, 0)
    for _ in range(cantidad):
        dia = min((d for d in libres if puestos[d] < libres[d]), key=lambda d: (puestos[d], d))
        puestos[dia] += 1
    return puestos


def _parejas(bloques: list[int], tam: int):
    """Grupos de `tam` bloques seguidos, del más temprano al más tardío; un bloque que no completa grupo queda libre."""
    grupos, corrida = [], []
    for b in bloques + [None]:
        if corrida and (b is None or b != corrida[-1] + BLOQUE_MIN):
            grupos += [corrida[i:i + tam] for i in range(0, len(corrida) - tam + 1, tam)]
            corrida = []
        if b is not None:
            corrida.append(b)
    return grupos


def _a_sesiones(nombre: str, dia: date, bloques: list[int], max_bloques: int):
    """Une bloques consecutivos en sesiones; si una corrida pasa de `max_bloques`, la parte en sesiones parejas
    (7 bloques con máximo 6 dan 4 + 3, no 6 + 1)."""
    corridas = []
    for b in bloques:
        if corridas and b == corridas[-1][-1] + BLOQUE_MIN:
            corridas[-1].append(b)
        else:
            corridas.append([b])
    sesiones = []
    for c in corridas:
        k = ceil(len(c) / max_bloques)
        base, extra, i = *divmod(len(c), k), 0
        for j in range(k):
            n = base + (j < extra)
            sesiones.append(Sesion(actividad=nombre, dia=dia, inicio=_hora(c[i]), fin=_hora(c[i] + n * BLOQUE_MIN)))
            i += n
    return sesiones


def _completar(faltan: int, ventana: list[date], libres: dict[date, list[int]], tomados: dict[date, list[int]]):
    """Ubica `faltan` bloques sueltos: pegados a lo ya tomado (el día menos cargado de la actividad) para no abrir una
    sesión de 30 min aparte y, si no hay, en cualquier hueco de la ventana. Devuelve los que no cupieron."""
    while faltan:
        propios = {d: set(tomados.get(d, ())) for d in ventana}
        pegados = [(len(propios[d]), d, b) for d in ventana for b in libres[d]
                   if b - BLOQUE_MIN in propios[d] or b + BLOQUE_MIN in propios[d]]
        sueltos = [(len(propios[d]), d, libres[d][0]) for d in ventana if libres[d]]
        elegido = min(pegados or sueltos, default=None)
        if elegido is None:
            break
        _, dia, bloque = elegido
        libres[dia].remove(bloque)
        tomados.setdefault(dia, []).append(bloque)
        faltan -= 1
    return faltan


def _corridas(bloques: list[int]):
    """Grupos de bloques seguidos (un hueco sin bloques corta el grupo)."""
    grupos: list[list[int]] = []
    for b in bloques:
        if grupos and b == grupos[-1][-1] + BLOQUE_MIN:
            grupos[-1].append(b)
        else:
            grupos.append([b])
    return grupos


def _ubicar_unica(total: int, ventana: list[date], libres: dict[date, list[int]], carga: dict[date, int], perfil: list[float]):
    """Hueco seguido de `total` bloques para la actividad completa: en el tramo que prefiere su perfil (o el vecino más
    cercano), el día menos cargado; a igual carga, el hueco que deja un resto par (así no queda media hora suelta) y,
    después, el más ajustado. Devuelve (día, bloques) o None si ningún día tiene un hueco así de largo."""
    tramos = _dividir_dias(ventana)
    preferido = _repartir_tramos(1, perfil).index(1)
    for t in sorted(range(TRAMOS), key=lambda j: (abs(j - preferido), -j)):
        opciones = [(carga[d], (len(c) - total) % 2, len(c), d, c[0]) for d in tramos[t] for c in _corridas(libres[d]) if len(c) >= total]
        if opciones:
            _, _, _, dia, inicio = min(opciones)
            return dia, [b for b in libres[dia] if inicio <= b < inicio + total * BLOQUE_MIN]
    return None


def _valida(n: int, max_bloques: int) -> bool:
    """Un grupo de `n` bloques seguidos de una actividad queda en sesiones de al menos 1 h (0 = no hay grupo)."""
    return n == 0 or (n >= SESION_MIN // BLOQUE_MIN and n // ceil(n / max_bloques) >= SESION_MIN // BLOQUE_MIN)


def _limpiar_cortas(colocado: dict[int, dict[date, list[int]]], libres0: dict[date, list[int]], fijas: set[int], max_bloques: int):
    """Quita las sesiones de media hora de las actividades que se pueden mover. Trabaja con cuántos bloques tiene cada
    actividad en cada hueco seguido del día (dentro de un hueco el orden da igual) y, para cada grupo que quedó corto, prueba:
      1. traer a ese hueco un bloque de otra sesión de la misma actividad, si el hueco tiene espacio;
      2. llevar el bloque a otro hueco donde la actividad ya estudia: si hay espacio, directo; si no, cambiándolo por un
         bloque de otra actividad que en los dos huecos quede en sesiones de al menos 1 h.
    Nunca cambia cuántas horas tiene cada actividad ni cuántas se estudian cada día."""
    huecos = [(d, c) for d in sorted(libres0) for c in _corridas(libres0[d])]
    hueco_de = {(d, b): k for k, (d, c) in enumerate(huecos) for b in c}
    x: dict[int, dict[int, int]] = {i: {} for i in colocado}
    for i, por_dia in colocado.items():
        for d, bloques in por_dia.items():
            for b in bloques:
                x[i][hueco_de[d, b]] = x[i].get(hueco_de[d, b], 0) + 1
    libre = [len(c) - sum(xi.get(k, 0) for xi in x.values()) for k, (_, c) in enumerate(huecos)]
    moviles = [i for i in sorted(x) if i not in fijas and sum(x[i].values()) > 1]
    tocados: set[int] = set()

    def mover(i: int, de: int, a: int):
        x[i][de] -= 1
        if not x[i][de]:
            del x[i][de]
        x[i][a] = x[i].get(a, 0) + 1
        tocados.update((de, a))

    def arreglar(i: int, k: int) -> bool:
        xi = x[i]
        if libre[k] and _valida(xi[k] + 1, max_bloques):
            traibles = [k3 for k3 in xi if k3 != k and _valida(xi[k3] - 1, max_bloques)]
            if traibles:
                k3 = min(traibles, key=lambda k3: (xi[k3] != 1, -xi[k3], k3))
                mover(i, k3, k)
                libre[k3] += 1
                libre[k] -= 1
                return True
        for k2 in sorted((k2 for k2 in xi if k2 != k and _valida(xi[k2] + 1, max_bloques)),
                         key=lambda k2: (xi[k2] != 1, k2)):
            if libre[k2]:
                mover(i, k, k2)
                libre[k] += 1
                libre[k2] -= 1
                return True
            for j in moviles:
                if j != i and k in x[j] and k2 in x[j] and _valida(x[j][k2] - 1, max_bloques) and _valida(x[j][k] + 1, max_bloques):
                    mover(i, k, k2)
                    mover(j, k2, k)
                    return True
        return False

    avanzo = True
    while avanzo:
        avanzo = False
        for i in moviles:
            for k in sorted(x[i]):
                if k in x[i] and not _valida(x[i][k], max_bloques) and arreglar(i, k):
                    avanzo = True

    for k in sorted(tocados):                                  # solo se vuelven a acomodar los huecos que cambiaron
        d, c = huecos[k]
        antes = {i: [b for b in colocado[i].get(d, []) if hueco_de[d, b] == k] for i in colocado}
        orden = sorted((i for i in x if k in x[i]), key=lambda i: (antes[i][0] if antes[i] else c[-1] + 1, i))
        for i in colocado:
            if d in colocado[i]:
                colocado[i][d] = [b for b in colocado[i][d] if hueco_de[d, b] != k]
        sitio = 0
        for i in orden:
            colocado[i].setdefault(d, []).extend(c[sitio:sitio + x[i][k]])
            sitio += x[i][k]
    return colocado


def _costo(plan: PlanEntrada, r: ResultadoAsignacion, max_bloques: int):
    """Con qué tan mal quedó un horario, de lo más grave a lo menos: horas sin ubicar, sesiones de menos de 1 h (salvo en
    actividades de media hora) y actividades de una sola sesión que quedaron partidas."""
    faltan = sum(x.horas_faltantes for x in r.actividades)
    cortas = partidas = 0
    for a in plan.actividades:
        propias = [s for s in r.sesiones if s.actividad == a.nombre]
        bloques = max(1, int(a.horas_estimadas * 60 / BLOQUE_MIN + 0.5))
        if bloques > 1:
            cortas += sum(_minutos(s.fin) - _minutos(s.inicio) < SESION_MIN for s in propias)
        if a.prioridad != Nivel.ALTA and bloques <= max_bloques and len(propias) > 1:
            partidas += 1
    return faltan, cortas, partidas


def asignar(plan: PlanEntrada, predicciones: list[Prediccion], fecha_inicio: date,
            ocupadas: list[Sesion] | None = None, respaldo: bool = True) -> ResultadoAsignacion:
    """Prueba dos órdenes de trabajo (por prioridad, y con las actividades de una sola sesión primero, que son las que más
    necesitan un hueco seguido) y se queda con el que deje el mejor horario. Si aun así queda algo sin ubicar y
    `respaldo` es True, prueba también sin las reglas de sesión única y de limpieza, y con bloques de 30 min, y se queda
    con lo que ubique más horas."""
    if len(predicciones) != len(plan.actividades):
        raise ValueError("debe haber una predicción por actividad, en el mismo orden")
    ocupadas = ocupadas or []
    max_bloques = plan.duracion_sesion // BLOQUE_MIN
    unidad = min(SESION_MIN // BLOQUE_MIN, max_bloques)  # bloques por pieza (2 = sesiones de 1 h o más)

    def costo(r: ResultadoAsignacion):
        return _costo(plan, r, max_bloques)

    candidatos = [_asignar(plan, predicciones, fecha_inicio, ocupadas, unidad, primero_unicas=pu) for pu in (False, True)]
    mejor = min(candidatos, key=costo)                         # a igual costo, el orden por prioridad
    if respaldo and costo(mejor)[0] > 0:
        extras = [_asignar(plan, predicciones, fecha_inicio, ocupadas, unidad, reglas=False)]
        if unidad > 1:
            extras += [_asignar(plan, predicciones, fecha_inicio, ocupadas, 1, reglas=False)]
            extras += [_asignar(plan, predicciones, fecha_inicio, ocupadas, 1, primero_unicas=pu) for pu in (False, True)]
        mejor = min([mejor] + extras, key=costo)
    return mejor


def _asignar(plan: PlanEntrada, predicciones: list[Prediccion], fecha_inicio: date,
             ocupadas: list[Sesion], unidad: int, primero_unicas: bool = False, reglas: bool = True) -> ResultadoAsignacion:
    libres = {d.dia: _bloques_dia(d, plan.max_horas_dia, [s for s in ocupadas if s.dia == d.dia])
              for d in plan.disponibilidad if d.dia >= fecha_inicio}
    libres0 = {d: list(b) for d, b in libres.items()}
    max_bloques = plan.duracion_sesion // BLOQUE_MIN
    colocado: dict[int, dict[date, list[int]]] = {}
    fijas: set[int] = set()                                    # actividades que quedaron en una sola sesión: no se mueven
    faltantes: dict[int, int] = {}
    totales: dict[int, int] = {}

    def es_unica(i: int):                                      # prioridad media o baja y cabe en una sesión
        a = plan.actividades[i]
        return reglas and a.prioridad != Nivel.ALTA and max(1, int(a.horas_estimadas * 60 / BLOQUE_MIN + 0.5)) <= max_bloques

    orden = _ordenar(plan.actividades, predicciones)
    if primero_unicas:
        orden.sort(key=lambda i: not es_unica(i))              # estable: dentro de cada grupo sigue el orden por prioridad

    for i in orden:
        a, p = plan.actividades[i], predicciones[i]
        total = max(1, int(a.horas_estimadas * 60 / BLOQUE_MIN + 0.5))
        totales[i] = total
        ventana = sorted(d for d, b in libres.items() if b and d < a.fecha_limite)

        unica = None
        if es_unica(i):                                        # una sola sesión
            carga = {d: len(libres0[d]) - len(libres[d]) for d in libres}
            unica = _ubicar_unica(total, ventana, libres, carga, p.perfil)
        if unica is not None:
            dia, bloques = unica
            colocado[i] = {dia: bloques}
            libres[dia] = [b for b in libres[dia] if b not in set(bloques)]
            fijas.add(i)
            faltantes[i] = 0
            continue

        tramos = _dividir_dias(ventana)
        piezas = {d: _parejas(libres[d], unidad) for d in ventana}
        enteras, resto = divmod(total, unidad)
        caben = [sum(len(piezas[d]) for d in t) for t in tramos]
        puestos, sin_ubicar = _ajustar_a_capacidad(_repartir_tramos(enteras, p.perfil), caben)

        tomados: dict[date, list[int]] = {}
        for tramo, n in zip(tramos, puestos):
            for dia, k in _repartir_en_dias(n, {d: len(piezas[d]) for d in tramo}).items():
                if k:
                    tomados[dia] = [b for pieza in piezas[dia][:k] for b in pieza]
                    libres[dia] = [b for b in libres[dia] if b not in set(tomados[dia])]
        faltantes[i] = _completar(sin_ubicar * unidad + resto, ventana, libres, tomados)
        colocado[i] = tomados

    if reglas and max_bloques >= SESION_MIN // BLOQUE_MIN:
        colocado = _limpiar_cortas(colocado, libres0, fijas, max_bloques)

    sesiones, resumen = [], []
    for i in _ordenar(plan.actividades, predicciones):
        a, p = plan.actividades[i], predicciones[i]
        for dia, bloques in sorted(colocado[i].items()):
            if bloques:
                sesiones += _a_sesiones(a.nombre, dia, sorted(bloques), max_bloques)
        faltan = faltantes[i]
        asignados = totales[i] - faltan
        estado = (EstadoAsignacion.ASIGNADA if faltan == 0 else
                  EstadoAsignacion.NO_ASIGNADA if asignados == 0 else EstadoAsignacion.PARCIAL)
        resumen.append(ResumenActividad(
            nombre=a.nombre, prioridad=p.prioridad, horas_solicitadas=a.horas_estimadas,
            horas_asignadas=asignados * BLOQUE_MIN / 60, horas_faltantes=faltan * BLOQUE_MIN / 60, estado=estado))

    sesiones.sort(key=lambda s: (s.dia, s.inicio))
    return ResultadoAsignacion(sesiones=sesiones, actividades=resumen)


## 3.1 Pruebas de las funciones auxiliares

Los valores esperados están calculados a mano. El caso de `_repartir_tramos(16, [0.45, 0.35, 0.20])` es el ejemplo del apartado 1.2.

In [7]:
import random
from datetime import date

H = lambda h, m=0: h * 60 + m


# _bloques_dia: franjas, redondeo, tope diario y ocupadas
def bl(*pares, tope=None, ocupadas=()):
    return _bloques_dia(dia("2026-10-26", *pares), tope, list(ocupadas))


assert bl(("18:00", "19:30")) == [H(18), H(18, 30), H(19)]
assert bl(("18:00", "18:50")) == [H(18)]                                       # 50 min -> 1 bloque
assert bl(("19:00", "20:00"), ("16:00", "17:00")) == [H(16), H(16, 30), H(19), H(19, 30)]   # franja más temprana primero
assert bl(("18:00", "21:00"), tope=1) == [H(18), H(18, 30)]
assert len(bl(("18:00", "21:00"), tope=2.75)) == 5                             # el tope también va a bloques
ocup = [Sesion(actividad="x", dia="2026-10-26", inicio="18:00", fin="19:00")]
assert bl(("18:00", "21:00"), ocupadas=ocup) == [H(19), H(19, 30), H(20), H(20, 30)]
assert bl(("18:00", "21:00"), tope=2, ocupadas=ocup) == [H(19), H(19, 30)]     # lo ocupado cuenta para el tope
assert bl(("18:00", "21:00"), tope=1, ocupadas=ocup) == []
for pares, tope in [((("11:00", "14:00"), ("16:00", "17:30")), None), ((("11:00", "14:00"), ("16:00", "17:30")), 4),
                    ((("18:00", "18:45"), ("19:00", "19:45")), None), ((("18:00", "18:20"),), None)]:
    assert len(bl(*pares, tope=tope)) / 2 == capacidad_dia(dia("2026-10-26", *pares), tope)   # igual que el validador

# _ordenar: prioridad, luego fecha límite, luego nombre
acts = [Actividad(nombre=n, prioridad="media", dificultad="media", fecha_limite=fl, horas_estimadas=1)
        for n, fl in [("C", "2026-11-03"), ("A", "2026-11-05"), ("B", "2026-11-03")]]
preds = [Prediccion(prioridad=q, perfil=[1/3] * 3) for q in (0.5, 0.5, 0.9)]
assert _ordenar(acts, preds) == [2, 0, 1]
assert _ordenar(acts, [Prediccion(prioridad=0.5, perfil=[1/3] * 3)] * 3) == [2, 0, 1]   # todo empatado: fecha y nombre (B < C)

# _repartir_tramos: mayor residuo
assert _repartir_tramos(16, [0.45, 0.35, 0.20]) == [7, 6, 3]       # 3 h 30, 3 h, 1 h 30
assert _repartir_tramos(8, [0.5, 0.3, 0.2]) == [4, 2, 2]
assert _repartir_tramos(8, [1, 0, 0]) == [8, 0, 0]
assert _repartir_tramos(1, [0.2, 0.5, 0.3]) == [0, 1, 0]
assert _repartir_tramos(8, [1/3] * 3) == [3, 3, 2]                  # residuos iguales: el tramo más temprano primero
random.seed(0)
for _ in range(5000):
    total = random.randint(1, 200)
    w = [random.random() for _ in range(3)]
    perfil = [x / sum(w) for x in w]
    r = _repartir_tramos(total, perfil)
    assert sum(r) == total and min(r) >= 0 and all(abs(x - q * total) < 1 for x, q in zip(r, perfil))

# _dividir_dias
ds = [date(2026, 10, 25 + i) for i in range(7)]
assert [len(t) for t in _dividir_dias(ds)] == [3, 2, 2]
assert [len(t) for t in _dividir_dias(ds[:6])] == [2, 2, 2]
assert [len(t) for t in _dividir_dias(ds[:3])] == [1, 1, 1]
assert [len(t) for t in _dividir_dias(ds[:2])] == [1, 1, 0]
assert [len(t) for t in _dividir_dias(ds[:1])] == [1, 0, 0]
assert [len(t) for t in _dividir_dias([])] == [0, 0, 0]
assert _dividir_dias(ds)[1] == ds[3:5]                              # tramos consecutivos

# _ajustar_a_capacidad
assert _ajustar_a_capacidad([7, 6, 3], [10, 10, 10]) == ([7, 6, 3], 0)
assert _ajustar_a_capacidad([4, 2, 2], [2, 12, 12]) == ([2, 4, 2], 0)     # el excedente del primero va al segundo
assert _ajustar_a_capacidad([2, 2, 4], [12, 12, 2]) == ([2, 4, 2], 0)     # el del último va al anterior
assert _ajustar_a_capacidad([3, 3, 2], [12, 2, 12]) == ([3, 2, 3], 0)     # a igual distancia, al posterior
assert _ajustar_a_capacidad([5, 0, 0], [2, 0, 3]) == ([2, 0, 3], 0)       # salta un tramo sin espacio
assert _ajustar_a_capacidad([4, 2, 2], [3, 0, 0]) == ([3, 0, 0], 5)       # lo que no cabe en ninguno queda sin ubicar
assert _ajustar_a_capacidad([8, 0, 0], [0, 0, 0]) == ([0, 0, 0], 8)

d1, d2, d3 = ds[:3]

# _parejas: grupos de bloques seguidos
assert _parejas([H(18), H(18, 30), H(19)], 2) == [[H(18), H(18, 30)]]                   # el bloque suelto de las 19:00 queda libre
assert _parejas([H(16), H(16, 30), H(17), H(17, 30), H(19), H(19, 30)], 2) == [[H(16), H(16, 30)], [H(17), H(17, 30)], [H(19), H(19, 30)]]
assert _parejas([H(18), H(19)], 2) == []                                               # no son consecutivos
assert _parejas([H(18), H(18, 30)], 1) == [[H(18)], [H(18, 30)]]
assert _parejas([], 2) == []

# _completar: el resto de 30 min
libres, tomados = {d1: [H(19), H(21)], d2: [H(18)]}, {d1: [H(18), H(18, 30)]}
assert _completar(1, [d1, d2], libres, tomados) == 0
assert tomados[d1] == [H(18), H(18, 30), H(19)] and libres[d1] == [H(21)]               # pegado a lo ya tomado
libres, tomados = {d1: [H(21)], d2: [H(18)]}, {d1: [H(18), H(18, 30)]}
assert _completar(1, [d1, d2], libres, tomados) == 0
assert tomados[d2] == [H(18)]                                                          # sin vecinos libres: el día menos cargado
libres, tomados = {d1: [H(21)], d2: [H(18)]}, {}
assert _completar(3, [d1, d2], libres, tomados) == 1                                   # lo que no cabe se devuelve
assert _completar(0, [], {}, {}) == 0

# _repartir_en_dias
assert _repartir_en_dias(7, {d1: 6, d2: 6}) == {d1: 4, d2: 3}
assert _repartir_en_dias(3, {d1: 6, d2: 1, d3: 6}) == {d1: 1, d2: 1, d3: 1}
assert _repartir_en_dias(5, {d1: 6, d2: 1}) == {d1: 4, d2: 1}             # el día pequeño se llena; el resto va al otro
assert _repartir_en_dias(2, {d1: 6, d2: 6, d3: 6}) == {d1: 1, d2: 1, d3: 0}
assert _repartir_en_dias(0, {}) == {}


# _a_sesiones
def ses(bloques, mx):
    return [(s.inicio.strftime("%H:%M"), s.fin.strftime("%H:%M")) for s in _a_sesiones("x", d1, bloques, mx)]


assert ses([H(18), H(18, 30), H(19)], 2) == [("18:00", "19:00"), ("19:00", "19:30")]
assert ses([H(18), H(18, 30), H(19), H(19, 30)], 8) == [("18:00", "20:00")]
assert ses([H(16), H(16, 30), H(19)], 8) == [("16:00", "17:00"), ("19:00", "19:30")]    # no cruza el hueco entre franjas
assert ses([H(18)], 1) == [("18:00", "18:30")]
assert ses([], 2) == []
assert ses([H(18 + i // 2, 30 * (i % 2)) for i in range(7)], 6) == [("18:00", "20:00"), ("20:00", "21:30")]   # 7 bloques: 4 + 3, no 6 + 1
assert ses([H(18 + i // 2, 30 * (i % 2)) for i in range(4)], 3) == [("18:00", "19:00"), ("19:00", "20:00")]   # 4 bloques con máximo 3: 2 + 2

# _corridas: un hueco entre bloques corta el grupo
assert _corridas([H(18), H(18, 30), H(20)]) == [[H(18), H(18, 30)], [H(20)]]
assert _corridas([H(18)]) == [[H(18)]]
assert _corridas([]) == []


def corrida(ini, n):
    return [H(ini) + 30 * i for i in range(n)]


# _ubicar_unica: un hueco seguido para toda la actividad
d4 = ds[3]
libres = {d1: corrida(18, 3), d2: corrida(18, 4), d3: corrida(18, 4)}
cero = dict.fromkeys(ds[:3], 0)
assert _ubicar_unica(2, ds[:3], libres, cero, [1, 0, 0]) == (d1, [H(18), H(18, 30)])       # el tramo que prefiere el perfil
assert _ubicar_unica(2, ds[:3], libres, cero, [0, 1, 0]) == (d2, [H(18), H(18, 30)])
assert _ubicar_unica(5, ds[:3], libres, cero, [1, 0, 0]) is None                            # ningún hueco seguido alcanza
assert _ubicar_unica(4, ds[:3], libres, cero, [1, 0, 0]) == (d2, corrida(18, 4))           # sin hueco en su tramo: el vecino
libres = {d1: corrida(18, 4), d2: corrida(18, 5), d3: corrida(18, 4), d4: corrida(18, 4)}
assert _ubicar_unica(3, ds[:4], libres, dict.fromkeys(ds[:4], 0), [1, 0, 0]) == (d2, corrida(18, 3))           # d1 dejaría media hora suelta
assert _ubicar_unica(3, ds[:4], libres, {d1: 0, d2: 4, d3: 0, d4: 0}, [1, 0, 0]) == (d1, corrida(18, 3))       # el día menos cargado va primero
libres = {d1: corrida(18, 2) + corrida(20, 4)}
assert _ubicar_unica(3, ds[:1], libres, {d1: 0}, [1, 0, 0]) == (d1, corrida(20, 3))                            # un hueco seguido, aunque no sea el primero
assert _ubicar_unica(1, [], {}, {}, [1, 0, 0]) is None

# _valida: un grupo de n bloques seguidos queda en sesiones de al menos 1 h
assert all(_valida(n, 6) for n in (0, 2, 3, 6, 7, 11))      # 7 con máximo 6 son 4 + 3; 13 son 5 + 4 + 4
assert not _valida(1, 6)
assert _valida(4, 2) and not _valida(3, 2) and not _valida(5, 2)          # con máximo de 1 h no hay sesiones de 1,5 h

# _limpiar_cortas: mover bloques entre huecos sin cambiar las horas de nadie
base = lambda: {d1: corrida(18, 3), d2: corrida(18, 2)}
# hay espacio en el otro hueco: el bloque suelto se va con las demás horas de su actividad
r = _limpiar_cortas({0: {d1: [H(18), H(18, 30)], d2: [H(18)]}}, base(), set(), 6)
assert r[0] == {d1: [H(18), H(18, 30), H(19)], d2: []}
# hay espacio en el hueco del bloque suelto: se trae un bloque de la sesión larga
r = _limpiar_cortas({0: {d1: [H(18), H(18, 30), H(19)], d2: [H(18)]}}, base(), set(), 6)
assert r[0] == {d1: [H(18), H(18, 30)], d2: [H(18), H(18, 30)]}
# no hay espacio: se cambia el bloque por uno de otra actividad que en los dos huecos queda en sesiones de al menos 1 h
llenos = {d1: corrida(18, 6), d2: corrida(18, 4)}
r = _limpiar_cortas({0: {d1: corrida(18, 3), d2: [H(18)]}, 1: {d1: [H(19, 30), H(20), H(20, 30)], d2: [H(18, 30), H(19), H(19, 30)]}}, llenos, set(), 6)
assert r[0] == {d1: corrida(18, 4), d2: []} and r[1] == {d1: [H(20), H(20, 30)], d2: corrida(18, 4)}
# las actividades de una sola sesión no se mueven
r = _limpiar_cortas({0: {d1: [H(18), H(18, 30)], d2: [H(18)]}}, base(), {0}, 6)
assert r[0] == {d1: [H(18), H(18, 30)], d2: [H(18)]}
# con sesiones de máximo 1 h no se forman sesiones de 1 h 30
r = _limpiar_cortas({0: {d1: [H(18), H(18, 30)], d2: [H(18)]}}, base(), set(), 2)
assert r[0] == {d1: [H(18), H(18, 30)], d2: [H(18)]}
# una actividad de media hora no se toca
r = _limpiar_cortas({0: {d1: [H(18)]}}, base(), set(), 6)
assert r[0] == {d1: [H(18)]}

# asignar: una predicción por actividad
plan_min = PlanEntrada(actividades=[dict(nombre="T", prioridad="alta", dificultad="alta", fecha_limite="2026-11-05", horas_estimadas=2)],
                       disponibilidad=[{"dia": "2026-11-01", "franjas": franjas(("18:00", "21:00"))}], duracion_sesion=60)
try:
    asignar(plan_min, [], date(2026, 11, 1))
    raise AssertionError("debía rechazar la falta de predicciones")
except ValueError as e:
    assert "una predicción por actividad" in str(e)

print("Funciones auxiliares: todas las pruebas pasaron")

Funciones auxiliares: todas las pruebas pasaron


# PASO 4. UN EJEMPLO COMPLETO

Dos actividades con 3 h de disponibilidad por día (18:00-21:00) y sesión máxima de 60 min. Primero pasan por el validador y luego por la asignación. Las predicciones se escriben a mano, como si las hubiera entregado el MLP.

* **Parcial de IA:** 8 h, 6 días, prioridad alta y dificultad alta -> `prioridad 0.87`, perfil `[0.45, 0.35, 0.20]` (adelantar el trabajo).
* **Taller:** 4 h, 5 días -> `prioridad 0.55`, perfil `[0.30, 0.40, 0.30]`.

In [8]:
from datetime import timedelta

import pandas as pd

INICIO = date(2026, 10, 25)  # domingo


def f(n):
    return (INICIO + timedelta(days=n)).isoformat()


def act(nombre, n_limite, horas, prioridad="media", dificultad="media"):
    return {"nombre": nombre, "prioridad": prioridad, "dificultad": dificultad, "fecha_limite": f(n_limite), "horas_estimadas": horas}


def dias(desde, hasta, pares=(("18:00", "21:00"),)):
    return [{"dia": f(n), "franjas": franjas(*pares)} for n in range(desde, hasta)]


def plan(actividades, disponibilidad, sesion=60, max_horas_dia=None):
    return {"actividades": actividades, "disponibilidad": disponibilidad, "duracion_sesion": sesion, "max_horas_dia": max_horas_dia}


def pr(prioridad, perfil):
    return Prediccion(prioridad=prioridad, perfil=perfil)


def tabla_sesiones(r):
    return pd.DataFrame([{"actividad": s.actividad, "día": f"{s.dia:%a %d} (n={(s.dia - INICIO).days})",
                          "inicio": f"{s.inicio:%H:%M}", "fin": f"{s.fin:%H:%M}"} for s in r.sesiones])


ejemplo = PlanEntrada(**plan([act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)], dias(0, 10)))
print("Validador:", validar(ejemplo, INICIO).estado.value)

r = asignar(ejemplo, [pr(0.87, [0.45, 0.35, 0.20]), pr(0.55, [0.30, 0.40, 0.30])], INICIO)
print(pd.DataFrame([a.model_dump(mode="json") for a in r.actividades]).to_string(index=False))
tabla_sesiones(r)

Validador: factible
       nombre  prioridad  horas_solicitadas  horas_asignadas  horas_faltantes   estado
Parcial de IA       0.87                8.0              8.0              0.0 asignada
       Taller       0.55                4.0              4.0              0.0 asignada


,actividad,día,inicio,fin
0,Parcial de IA,Sun 25 (n=0),18:00,19:00
1,Parcial de IA,Sun 25 (n=0),19:00,20:00
2,Taller,Sun 25 (n=0),20:00,21:00
3,Parcial de IA,Mon 26 (n=1),18:00,19:00
4,Parcial de IA,Mon 26 (n=1),19:00,20:00
5,Parcial de IA,Tue 27 (n=2),18:00,19:00
6,Parcial de IA,Tue 27 (n=2),19:00,20:00
7,Taller,Tue 27 (n=2),20:00,21:00
8,Parcial de IA,Wed 28 (n=3),18:00,19:00
9,Taller,Wed 28 (n=3),19:00,20:00


**Observaciones**

1. El validador deja pasar el plan (`factible`) y la asignación ubica las 12 h pedidas: las dos actividades quedan `asignada`.
2. **Parcial de IA** (prioridad 0.87) escoge primero, por eso ocupa el inicio de cada día (18:00). Sus piezas por tramo son las del apartado 1.2: **4 h** en n=0 y n=1 (2 h cada día), **3 h** en n=2 y n=3 (2 h y 1 h) y **1 h** en n=4.
3. **Taller** (prioridad 0.55) se acomoda en lo que queda de cada día y reparte sus 4 h en cuatro sesiones de 1 h (n=0, n=2, n=3 y n=4). Como vence antes (n=5), no trabaja el viernes.
4. Ninguna sesión pasa de 60 min (`duracion_sesion`) y ninguna dura menos de 1 h: ya no hay medias horas sueltas.
5. `ResultadoAsignacion` trae exactamente lo que el front necesita para pintar el calendario y lo que el LLM redactor y el validador de salida necesitan para hablar del horario sin inventar nada.

# PASO 5. CASOS DE PRUEBA

Cada caso guarda **entrada, salida esperada y salida lograda**, y esta tabla alimenta la sección 7 del informe. Los casos son datos (`CASOS`), así que después se reutilizan con `pytest.mark.parametrize` sin reescribirlos.

Todos usan `INICIO` = domingo 25 de octubre de 2026 y, salvo que se diga otra cosa, 3 h de disponibilidad por día (18:00-21:00) y sesión de 60 min. `n` es el día respecto al inicio (0 = domingo). La entrada incluye las predicciones del MLP, escritas a mano.

## 5.1 Reglas que todo resultado debe cumplir

`verificar` se ejecuta en **todos** los casos y en las pruebas aleatorias. Comprueba el objetivo del proyecto de **0 sesiones fuera de la disponibilidad**, además de las fechas, la duración de las sesiones, el máximo diario y la coherencia del resumen.

In [9]:
def verificar(plan_dict, r, fecha_inicio, ocupadas=()):
    p = PlanEntrada(**plan_dict)
    por_nombre = {a.nombre: a for a in p.actividades}
    franjas_de = {d.dia: d.franjas for d in p.disponibilidad}
    horas = dict.fromkeys(por_nombre, 0.0)
    for s in r.sesiones:
        a = por_nombre[s.actividad]
        ini, fin = _minutos(s.inicio), _minutos(s.fin)
        assert fecha_inicio <= s.dia < a.fecha_limite, f"fuera de fechas: {s}"
        assert BLOQUE_MIN <= fin - ini <= p.duracion_sesion and (fin - ini) % BLOQUE_MIN == 0, f"duración inválida: {s}"
        for b in range(ini, fin, BLOQUE_MIN):   # cada bloque debe estar dentro de una franja
            assert any(_minutos(fr.inicio) <= b and b + BLOQUE_MIN <= _minutos(fr.fin) for fr in franjas_de.get(s.dia, [])), f"fuera de disponibilidad: {s}"
        horas[s.actividad] += (fin - ini) / 60
    todas = sorted(list(r.sesiones) + list(ocupadas), key=lambda s: (s.dia, s.inicio))
    for x, y in zip(todas, todas[1:]):
        assert x.dia != y.dia or x.fin <= y.inicio, f"sesiones traslapadas: {x} / {y}"
    if p.max_horas_dia is not None:
        for d in {s.dia for s in todas}:
            total = sum((_minutos(s.fin) - _minutos(s.inicio)) / 60 for s in todas if s.dia == d)
            assert total <= p.max_horas_dia + 1e-9, f"supera el máximo diario el {d}"
    assert len(r.actividades) == len(p.actividades)
    for a in r.actividades:
        assert abs(horas[a.nombre] - a.horas_asignadas) < 1e-9, f"el resumen no coincide con las sesiones: {a.nombre}"
        assert (a.estado == EstadoAsignacion.ASIGNADA) == (a.horas_faltantes == 0)
        assert (a.estado == EstadoAsignacion.NO_ASIGNADA) == (a.horas_asignadas == 0)
    assert all(x.prioridad >= y.prioridad for x, y in zip(r.actividades, r.actividades[1:]))   # resumen en orden de prioridad

## 5.2 Casos

In [10]:
def hh(h):
    return f"{h:g}"


def esp(estados, horas_dia=None, sesiones=None):
    return {"estados": estados, "horas_dia": horas_dia, "sesiones": sesiones}


UNIF = [1/3] * 3
OC = Sesion(actividad="Fijada", dia=f(0), inicio="18:00", fin="19:00")

CASOS = [
    {"nombre": "Reparto del ejemplo (8 h)", "resumen": "8 h, 6 días, perfil [0.45, 0.35, 0.20]: 4 h / 3 h / 1 h",
     "entrada": plan([act("Parcial", 6, 8, "alta", "alta")], dias(0, 10)), "pred": [pr(0.87, [0.45, 0.35, 0.20])],
     "esperado": esp({"Parcial": "asignada"}, {0: 2, 1: 2, 2: 2, 3: 1, 4: 1})},
    {"nombre": "Manda la prioridad, no el orden de la lista", "resumen": "dos de 3 h compiten por 18:00-20:00; 'Taller B' va primero en la lista pero con prioridad 0.9",
     "entrada": plan([act("Taller B", 3, 3), act("Informe A", 3, 3)], dias(0, 3, (("18:00", "20:00"),))), "pred": [pr(0.9, UNIF), pr(0.4, UNIF)],
     "esperado": esp({"Taller B": "asignada", "Informe A": "asignada"}, {0: 2, 1: 2, 2: 2},
                     [(t, n, i, e) for n in range(3) for t, i, e in (("Taller B", "18:00", "19:00"), ("Informe A", "19:00", "20:00"))])},
    {"nombre": "Primer tramo sin espacio", "resumen": "días n=0 y n=1 de solo 30 min (no cabe una pieza de 1 h); el primer tramo entrega todo al segundo",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 2, (("18:00", "18:30"),)) + dias(2, 10)), "pred": [pr(0.7, [0.5, 0.3, 0.2])],
     "esperado": esp({"Proyecto": "asignada"}, {2: 2, 3: 1, 4: 1})},
    {"nombre": "Último tramo sin espacio", "resumen": "días n=4 y n=5 de solo 30 min; el último tramo entrega al segundo",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 4) + dias(4, 6, (("18:00", "18:30"),)) + dias(6, 10)), "pred": [pr(0.7, [0.2, 0.3, 0.5])],
     "esperado": esp({"Proyecto": "asignada"}, {0: 1, 2: 2, 3: 1})},
    {"nombre": "Tramo del medio sin espacio", "resumen": "días n=2 y n=3 de solo 30 min; el tramo del medio entrega al posterior, no al anterior",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 2) + dias(2, 4, (("18:00", "18:30"),)) + dias(4, 10)), "pred": [pr(0.7, [0.3, 0.4, 0.3])],
     "esperado": esp({"Proyecto": "asignada"}, {0: 1, 4: 2, 5: 1})},
    {"nombre": "Solo cabe una parte", "resumen": "6 h de trabajo y 3 h libres (1 h por día): se asigna lo que cabe",
     "entrada": plan([act("Tesis", 3, 6)], dias(0, 3, (("18:00", "19:00"),))), "pred": [pr(0.8, UNIF)],
     "esperado": esp({"Tesis": "parcial (faltan 3 h)"}, {0: 1, 1: 1, 2: 1})},
    {"nombre": "Sin espacio para la de menor prioridad", "resumen": "3 h libres; 'Examen' (prioridad 0.9) las toma todas y 'Lectura' queda fuera",
     "entrada": plan([act("Examen", 3, 3), act("Lectura", 3, 2)], dias(0, 3, (("18:00", "19:00"),))), "pred": [pr(0.9, UNIF), pr(0.3, UNIF)],
     "esperado": esp({"Examen": "asignada", "Lectura": "no_asignada (faltan 2 h)"}, {0: 1, 1: 1, 2: 1})},
    {"nombre": "Máximo diario de 1 h", "resumen": "4 h en 5 días de 3 h libres, pero máximo de 1 h por día: cuatro sesiones de 1 h",
     "entrada": plan([act("Proyecto", 5, 4)], dias(0, 10), max_horas_dia=1), "pred": [pr(0.6, UNIF)],
     "esperado": esp({"Proyecto": "asignada"}, {0: 1, 1: 1, 2: 1, 4: 1})},
    {"nombre": "Sesiones de máximo 90 min", "resumen": "4 h el mismo día, franja 18:00-22:00: 90 + 90 + 60 min",
     "entrada": plan([act("Informe", 3, 4)], dias(0, 10, (("18:00", "22:00"),)), sesion=90), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 4}, [("Informe", 0, "18:00", "19:30"), ("Informe", 0, "19:30", "21:00"), ("Informe", 0, "21:00", "22:00")])},
    {"nombre": "Dos franjas en el mismo día", "resumen": "franjas 16:00-17:30 y 19:00-20:30; 2 h en dos sesiones de 1 h, una por franja",
     "entrada": plan([act("Informe", 3, 2)], dias(0, 10, (("16:00", "17:30"), ("19:00", "20:30"))), sesion=90), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 2}, [("Informe", 0, "16:00", "17:00"), ("Informe", 0, "19:00", "20:00")])},
    {"nombre": "Franja que no es múltiplo de 30 min", "resumen": "franja 18:00-19:50 aporta 3 bloques: una pieza de 1 h; la otra hora va al día siguiente",
     "entrada": plan([act("Informe", 3, 2)], dias(0, 10, (("18:00", "19:50"),)), sesion=120), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 1, 1: 1}, [("Informe", 0, "18:00", "19:00"), ("Informe", 1, "18:00", "19:00")])},
    {"nombre": "Replanificación: lo ocupado se respeta", "resumen": "hay una sesión fija el día n=0 de 18:00 a 19:00", "ocupadas": [OC],
     "entrada": plan([act("Taller", 3, 2)], dias(0, 10)), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 2}, [("Taller", 0, "19:00", "20:00"), ("Taller", 0, "20:00", "21:00")])},
    {"nombre": "Lo ocupado cuenta para el máximo diario", "resumen": "igual al anterior con máximo de 2 h: queda 1 h el día n=0 y el resto pasa al siguiente", "ocupadas": [OC],
     "entrada": plan([act("Taller", 3, 2)], dias(0, 10), max_horas_dia=2), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 1, 1: 1}, [("Taller", 0, "19:00", "20:00"), ("Taller", 1, "18:00", "19:00")])},
    {"nombre": "Ventana de 2 días (replanificación)", "resumen": "3 h con solo n=0 y n=1; el tercer tramo no tiene días y entrega al segundo",
     "entrada": plan([act("Quiz", 2, 3)], dias(0, 10)), "pred": [pr(0.6, [0.4, 0.3, 0.3])],
     "esperado": esp({"Quiz": "asignada"}, {0: 1, 1: 2})},
    {"nombre": "Fecha límite vencida", "resumen": "'Vieja' vence en n=0 (sin días de trabajo); 'Taller' se asigna normal",
     "entrada": plan([act("Vieja", 0, 2), act("Taller", 5, 2)], dias(0, 10)), "pred": [pr(0.9, UNIF), pr(0.5, UNIF)],
     "esperado": esp({"Vieja": "no_asignada (faltan 2 h)", "Taller": "asignada"}, {0: 1, 2: 1})},
    {"nombre": "Disponibilidad anterior al inicio", "resumen": "hay disponibilidad desde n=-2; se ignora",
     "entrada": plan([act("Taller", 6, 2)], dias(-2, 10)), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Taller": "asignada"}, {0: 1, 2: 1})},
    {"nombre": "Horas que no son múltiplo de 30 min", "resumen": "2.2 h se redondean a 2 h (4 bloques)",
     "entrada": plan([act("Taller", 3, 2.2)], dias(0, 10)), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 2})},
    {"nombre": "Sesiones de al menos 1 h", "resumen": "2.5 h con sesión de máximo 2 h: 1 h 30 y 1 h, sin sesiones sueltas de 30 min",
     "entrada": plan([act("Informe", 6, 2.5)], dias(0, 10), sesion=120), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Informe": "asignada"}, {0: 1.5, 2: 1}, [("Informe", 0, "18:00", "19:30"), ("Informe", 2, "18:00", "19:00")])},
    {"nombre": "Media hora solo si la actividad es de media hora", "resumen": "0.5 h: una única sesión de 30 min",
     "entrada": plan([act("Lectura", 3, 0.5)], dias(0, 10)), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Lectura": "asignada"}, {0: 0.5}, [("Lectura", 0, "18:00", "18:30")])},
    {"nombre": "Sesión máxima de 30 min", "resumen": "con duracion_sesion de 30 min la pieza es de 30 min: el máximo manda sobre el mínimo",
     "entrada": plan([act("Informe", 6, 1)], dias(0, 10), sesion=30), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Informe": "asignada"}, {0: 0.5, 2: 0.5}, [("Informe", 0, "18:00", "18:30"), ("Informe", 2, "18:00", "18:30")])},
    {"nombre": "Con piezas de 1 h no cabe todo: solo la media hora que hace falta",
     "resumen": "'A0' (2 h, vence n=4) y 'A1' (2.5 h, vence n=3) en días de 1 h, 2 h, 1 h y 1 h: con piezas de 1 h sobran 30 min, y el respaldo parte solo la media hora que hace falta",
     "entrada": plan([act("A0", 4, 2), act("A1", 3, 2.5)],
                     [{"dia": f(n), "franjas": franjas(("18:00", fin))} for n, fin in enumerate(["19:00", "20:00", "19:00", "19:00"])], sesion=90),
     "pred": [pr(0.2, [0.5, 1/3, 1/6]), pr(0.0, UNIF)],
     "esperado": esp({"A0": "asignada", "A1": "asignada"}, {0: 1, 1: 2, 2: 0.5, 3: 1},
                     [("A1", 0, "18:00", "19:00"), ("A0", 1, "18:00", "19:00"), ("A1", 1, "19:00", "20:00"), ("A1", 2, "18:00", "18:30"),
                      ("A0", 3, "18:00", "19:00")])},
    {"nombre": "Prioridad media que cabe en una sesión: una sola sesión", "resumen": "'Lab' de 2 h, prioridad media, sesión máxima de 3 h: una sesión de 2 h, no dos de 1 h",
     "entrada": plan([act("Lab", 6, 2)], dias(0, 10), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Lab": "asignada"}, {0: 2}, [("Lab", 0, "18:00", "20:00")])},
    {"nombre": "Prioridad baja que cabe en una sesión: una sola sesión", "resumen": "'Quiz' de 3 h, prioridad baja, sesión máxima de 3 h: una sesión de 3 h",
     "entrada": plan([act("Quiz", 6, 3, "baja", "baja")], dias(0, 10), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Quiz": "asignada"}, {0: 3}, [("Quiz", 0, "18:00", "21:00")])},
    {"nombre": "Sesión única: manda el perfil, no el primer día", "resumen": "igual al anterior, con perfil [0, 0, 1]: la sesión va al último tramo (n=4)",
     "entrada": plan([act("Lab", 6, 2)], dias(0, 10), sesion=180), "pred": [pr(0.5, [0, 0, 1])],
     "esperado": esp({"Lab": "asignada"}, {4: 2}, [("Lab", 4, "18:00", "20:00")])},
    {"nombre": "Prioridad alta se reparte aunque quepa en una sesión", "resumen": "'Parcial' de 2 h, prioridad alta, sesión de 3 h: se adelanta en dos sesiones de 1 h",
     "entrada": plan([act("Parcial", 6, 2, "alta", "alta")], dias(0, 10), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Parcial": "asignada"}, {0: 1, 2: 1}, [("Parcial", 0, "18:00", "19:00"), ("Parcial", 2, "18:00", "19:00")])},
    {"nombre": "Prioridad media que no cabe en una sesión se reparte", "resumen": "'Informe' de 4 h, prioridad media, sesión máxima de 3 h: no cabe en una sesión, se reparte en cuatro de 1 h",
     "entrada": plan([act("Informe", 6, 4)], dias(0, 10), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Informe": "asignada"}, {0: 1, 1: 1, 2: 1, 4: 1})},
    {"nombre": "Sin un hueco seguido, la sesión única se parte", "resumen": "días de 1 h y 'Lab' de 2 h: ningún día tiene 2 h seguidas, se parte en dos sesiones de 1 h antes que dejarla sin ubicar",
     "entrada": plan([act("Lab", 6, 2)], dias(0, 10, (("18:00", "19:00"),)), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Lab": "asignada"}, {0: 1, 2: 1}, [("Lab", 0, "18:00", "19:00"), ("Lab", 2, "18:00", "19:00")])},
    {"nombre": "Las de una sola sesión se ubican antes (necesitan un hueco seguido)", "resumen": "3 h de 'Parcial' (alta) y 3 h de 'Quiz' (media) en 4 días de 3 h: 'Quiz' va primero y toma un día entero; 'Parcial' se reparte en los demás",
     "entrada": plan([act("Parcial", 3, 3, "alta", "alta"), act("Quiz", 3, 3)], dias(0, 4), sesion=180), "pred": [pr(0.7, UNIF), pr(0.3, UNIF)],
     "esperado": esp({"Parcial": "asignada", "Quiz": "asignada"}, {0: 3, 1: 1, 2: 2},
                     [("Quiz", 0, "18:00", "21:00"), ("Parcial", 1, "18:00", "19:00"), ("Parcial", 2, "18:00", "20:00")])},
    {"nombre": "Sesión única: a igual carga, el hueco que no deja media hora suelta", "resumen": "'Lab' de 2 h; el día n=0 tiene 3 h 30 (dejaría media hora suelta) y el n=1 tiene 3 h: elige el n=1",
     "entrada": plan([act("Lab", 6, 2)], dias(0, 1, (("18:00", "21:30"),)) + dias(1, 10, (("18:00", "21:00"),)), sesion=180), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Lab": "asignada"}, {1: 2}, [("Lab", 1, "18:00", "20:00")])},
]


def ejecutar(caso):
    p = PlanEntrada(**caso["entrada"])
    ocupadas = caso.get("ocupadas", [])
    r = asignar(p, caso["pred"], INICIO, ocupadas)
    verificar(caso["entrada"], r, INICIO, ocupadas)
    estados = {a.nombre: a.estado.value + ("" if a.estado.value == "asignada" else f" (faltan {hh(a.horas_faltantes)} h)") for a in r.actividades}
    horas_dia = {}
    for s in r.sesiones:
        n = (s.dia - INICIO).days
        horas_dia[n] = horas_dia.get(n, 0) + (_minutos(s.fin) - _minutos(s.inicio)) / 60
    sesiones = [(s.actividad, (s.dia - INICIO).days, f"{s.inicio:%H:%M}", f"{s.fin:%H:%M}") for s in r.sesiones]
    return esp(estados, dict(sorted(horas_dia.items())), sesiones)


def coincide(esperado, lograda):
    return (esperado["estados"] == lograda["estados"]
            and esperado["horas_dia"] == lograda["horas_dia"]
            and (esperado["sesiones"] is None or esperado["sesiones"] == lograda["sesiones"]))


def texto(s):
    estados = ", ".join(f"{n}: {e}" for n, e in s["estados"].items())
    horas = " ".join(f"n{n}={hh(h)}" for n, h in s["horas_dia"].items())
    return f"{estados} | horas/día: {horas}"


filas = []
for c in CASOS:
    lograda = ejecutar(c)
    filas.append({"Caso": c["nombre"], "Entrada": c["resumen"], "Esperada": texto(c["esperado"]),
                  "Lograda": texto(lograda), "OK": "sí" if coincide(c["esperado"], lograda) else "NO"})

pd.set_option("display.max_colwidth", 120)
tabla = pd.DataFrame(filas)
print(f"{(tabla['OK'] == 'sí').sum()} de {len(tabla)} casos coinciden con lo esperado")
tabla

29 de 29 casos coinciden con lo esperado


,Caso,Entrada,Esperada,Lograda,OK
0,Reparto del ejemplo (8 h),"8 h, 6 días, perfil [0.45, 0.35, 0.20]: 4 h / 3 h / 1 h",Parcial: asignada | horas/día: n0=2 n1=2 n2=2 n3=1 n4=1,Parcial: asignada | horas/día: n0=2 n1=2 n2=2 n3=1 n4=1,sí
1,"Manda la prioridad, no el orden de la lista",dos de 3 h compiten por 18:00-20:00; 'Taller B' va primero en la lista pero con prioridad 0.9,"Taller B: asignada, Informe A: asignada | horas/día: n0=2 n1=2 n2=2","Taller B: asignada, Informe A: asignada | horas/día: n0=2 n1=2 n2=2",sí
2,Primer tramo sin espacio,días n=0 y n=1 de solo 30 min (no cabe una pieza de 1 h); el primer tramo entrega todo al segundo,Proyecto: asignada | horas/día: n2=2 n3=1 n4=1,Proyecto: asignada | horas/día: n2=2 n3=1 n4=1,sí
3,Último tramo sin espacio,días n=4 y n=5 de solo 30 min; el último tramo entrega al segundo,Proyecto: asignada | horas/día: n0=1 n2=2 n3=1,Proyecto: asignada | horas/día: n0=1 n2=2 n3=1,sí
4,Tramo del medio sin espacio,"días n=2 y n=3 de solo 30 min; el tramo del medio entrega al posterior, no al anterior",Proyecto: asignada | horas/día: n0=1 n4=2 n5=1,Proyecto: asignada | horas/día: n0=1 n4=2 n5=1,sí
5,Solo cabe una parte,6 h de trabajo y 3 h libres (1 h por día): se asigna lo que cabe,Tesis: parcial (faltan 3 h) | horas/día: n0=1 n1=1 n2=1,Tesis: parcial (faltan 3 h) | horas/día: n0=1 n1=1 n2=1,sí
6,Sin espacio para la de menor prioridad,3 h libres; 'Examen' (prioridad 0.9) las toma todas y 'Lectura' queda fuera,"Examen: asignada, Lectura: no_asignada (faltan 2 h) | horas/día: n0=1 n1=1 n2=1","Examen: asignada, Lectura: no_asignada (faltan 2 h) | horas/día: n0=1 n1=1 n2=1",sí
7,Máximo diario de 1 h,"4 h en 5 días de 3 h libres, pero máximo de 1 h por día: cuatro sesiones de 1 h",Proyecto: asignada | horas/día: n0=1 n1=1 n2=1 n4=1,Proyecto: asignada | horas/día: n0=1 n1=1 n2=1 n4=1,sí
8,Sesiones de máximo 90 min,"4 h el mismo día, franja 18:00-22:00: 90 + 90 + 60 min",Informe: asignada | horas/día: n0=4,Informe: asignada | horas/día: n0=4,sí
9,Dos franjas en el mismo día,"franjas 16:00-17:30 y 19:00-20:30; 2 h en dos sesiones de 1 h, una por franja",Informe: asignada | horas/día: n0=2,Informe: asignada | horas/día: n0=2,sí


In [11]:
assert (tabla["OK"] == "sí").all(), tabla[tabla["OK"] == "NO"]

# El respaldo: con piezas de 1 h sobra media hora; con bloques de 30 min cabe todo
caso = next(c for c in CASOS if c["nombre"].startswith("Con piezas de 1 h"))
p = PlanEntrada(**caso["entrada"])
assert sum(a.horas_faltantes for a in _asignar(p, caso["pred"], INICIO, [], 2).actividades) == 0.5
assert sum(a.horas_faltantes for a in asignar(p, caso["pred"], INICIO, respaldo=False).actividades) == 0.5
assert sum(a.horas_faltantes for a in asignar(p, caso["pred"], INICIO).actividades) == 0

# _costo: horas sin ubicar, sesiones cortas, sesiones únicas partidas
d = plan([act("Lab", 6, 2), act("Informe", 6, 4)], dias(0, 10), sesion=180)
p = PlanEntrada(**d)
bien = asignar(p, [pr(0.5, UNIF), pr(0.4, UNIF)], INICIO)
assert _costo(p, bien, 6) == (0, 0, 0)
partida = bien.model_copy(update={"sesiones": [s for s in bien.sesiones if s.actividad != "Lab"] + [
    Sesion(actividad="Lab", dia=f(0), inicio="18:00", fin="19:00"), Sesion(actividad="Lab", dia=f(1), inicio="18:00", fin="18:30")]})
assert _costo(p, partida, 6) == (0, 1, 1)                                  # una sesión de 30 min y una actividad de una sola sesión partida
sin_ubicar = bien.model_copy(update={"actividades": [x.model_copy(update={"horas_faltantes": 1.5}) for x in bien.actividades]})
assert _costo(p, sin_ubicar, 6)[0] == 3


**Observaciones**

1. Los tres casos de tramo sin espacio cubren las tres posiciones posibles: el excedente del primer tramo pasa al segundo, el del último pasa al anterior y el del medio pasa al **posterior** (la regla de desempate). Como un hueco de 30 min no admite una pieza de 1 h, esos días quedan sin trabajo y las piezas se van a los tramos vecinos.
2. En los dos casos de espacio insuficiente la actividad no se fuerza: queda `parcial` con las horas faltantes, o `no_asignada`. La prioridad decide quién se queda con el espacio (`Examen` sobre `Lectura`).
3. En "Manda la prioridad" la lista trae primero a la actividad más prioritaria, pero el caso se diseñó para que el orden de la lista no importe: lo que cuenta es la prioridad del MLP.
4. Los casos de replanificación comprueban que lo ya ocupado se respeta y que cuenta para el máximo diario, y que una ventana de menos de 3 días sigue funcionando (el tramo sin días entrega a su vecino).
5. `verificar` corrió dentro de cada caso: en los 29 hay **0 sesiones fuera de la disponibilidad**, ninguna en o después de la fecha límite ni antes del inicio, ninguna traslapada y ningún día sobre el máximo.
6. Cuatro casos son los de la sesión mínima: 2 h 30 se reparten en 1 h 30 y 1 h (la media hora se pega a una sesión, no queda suelta), una actividad de 0.5 h es la única que deja una sesión de 30 min, `duracion_sesion` de 30 min manda sobre el mínimo, y en "Con piezas de 1 h no cabe todo" las piezas de 1 h dejan 30 min sin ubicar, por lo que `asignar` usa el respaldo y parte **solo la media hora que hace falta**: las demás sesiones siguen siendo de 1 h.
7. Los ocho últimos casos son los de la sesión única: una actividad de prioridad media o baja que cabe en una sesión queda en un solo bloque (2 h, 3 h), el perfil decide en qué tramo cae, la prioridad alta se reparte aunque quepa, si no cabe en una sesión se reparte, si ningún día tiene un hueco seguido se parte antes que dejarla sin ubicar, las de sesión única se ubican antes que las demás porque necesitan un hueco seguido, y a igual carga se prefiere el hueco que no deja media hora suelta.


## 5.3 Plan ajustado de una prueba real

El caso que destapó el problema de los bloques de 30 min: 5 actividades, 34 h por ubicar en 34,5 h libres y días que empiezan a las 12:30, 13:30 y 15:30 (dejan medias horas sueltas), con sesión máxima de 3 h.

| Actividad | Horas | Prioridad | Límite |
|---|---|---|---|
| Proyecto ARSW | 20 | alta | n=6 |
| Proyecto PRI2 | 5 | media | n=4 |
| lab FDSI | 2 | media | n=6 |
| Quiz ARSW | 3 | media (dificultad baja) | n=3 |
| Lab IAIA | 4 | media | n=8 |

Se prueba con 4 perfiles y 3 órdenes de prioridad (12 combinaciones). Se exige: todo asignado, **ninguna sesión de menos de 1 h** y que `lab FDSI` y `Quiz ARSW` (prioridad media o baja que caben en una sesión) queden en **una sola sesión**. Una de las 12 combinaciones (prioridades `(0.6, 0.3, 0.9, 0.7, 0.1)` con el perfil `[0.5, 0.3, 0.2]`) deja 1 h sin ubicar porque el validador solo mira la capacidad total y no el orden en que se van llenando los días; la capa de órdenes lo resuelve adelantando la actividad incompleta y volviendo a asignar.


In [12]:
def plan_captura(sesion=180):
    disponibilidad = [{"dia": f(n), "franjas": franjas((i, fin))}
                      for n, i, fin in [(0, "15:00", "21:00"), (1, "18:00", "21:00"), (2, "12:30", "21:00"), (3, "13:30", "21:00"),
                                        (4, "18:00", "22:00"), (5, "15:30", "21:00")]]
    actividades = [act("Proyecto ARSW", 6, 20, "alta", "alta"), act("Proyecto PRI2", 4, 5), act("lab FDSI", 6, 2),
                   act("Quiz ARSW", 3, 3, "media", "baja"), act("Lab IAIA", 8, 4)]
    return plan(actividades, disponibilidad, sesion)


def duracion(s):
    return _minutos(s.fin) - _minutos(s.inicio)


d = plan_captura()
p = PlanEntrada(**d)
print("Validador:", validar(p, INICIO).estado.value)

PERFILES = [UNIF, [0.5, 0.3, 0.2], [0.2, 0.3, 0.5], [0.2, 0.6, 0.2]]
PRIORIDADES = [(0.9, 0.5, 0.5, 0.5, 0.5), (0.2, 0.9, 0.5, 0.4, 0.8), (0.6, 0.3, 0.9, 0.7, 0.1)]
completos = cortas_total = 0
for perfil in PERFILES:
    for prioridades in PRIORIDADES:
        r = asignar(p, [pr(x, perfil) for x in prioridades], INICIO, respaldo=False)
        if sum(a.horas_faltantes for a in r.actividades):
            print(f"  perfil {perfil}, prioridades {prioridades}: faltan {sum(a.horas_faltantes for a in r.actividades):g} h")
            continue
        verificar(d, r, INICIO)
        completos += 1
        cortas_total += sum(duracion(s) < SESION_MIN for s in r.sesiones)
        for nombre in ("lab FDSI", "Quiz ARSW"):
            assert len([s for s in r.sesiones if s.actividad == nombre]) == 1, (perfil, prioridades, nombre)
print(f"{completos} de {len(PERFILES) * len(PRIORIDADES)} combinaciones quedan completas; sesiones de menos de 1 h en ellas: {cortas_total}")

r = asignar(p, [pr(x, UNIF) for x in PRIORIDADES[0]], INICIO)
tabla_sesiones(r).assign(duración=[f"{duracion(s) / 60:g} h" for s in r.sesiones])


Validador: factible
  perfil [0.5, 0.3, 0.2], prioridades (0.6, 0.3, 0.9, 0.7, 0.1): faltan 1 h
11 de 12 combinaciones quedan completas; sesiones de menos de 1 h en ellas: 0


,actividad,día,inicio,fin,duración
0,Proyecto ARSW,Sun 25 (n=0),15:00,17:00,2 h
1,Proyecto ARSW,Sun 25 (n=0),17:00,19:00,2 h
2,Proyecto PRI2,Sun 25 (n=0),19:00,21:00,2 h
3,Proyecto ARSW,Mon 26 (n=1),18:00,21:00,3 h
4,Proyecto ARSW,Tue 27 (n=2),12:30,15:00,2.5 h
5,Proyecto ARSW,Tue 27 (n=2),15:00,17:00,2 h
6,Quiz ARSW,Tue 27 (n=2),17:00,20:00,3 h
7,Proyecto PRI2,Tue 27 (n=2),20:00,21:00,1 h
8,Proyecto ARSW,Wed 28 (n=3),13:30,16:30,3 h
9,Proyecto PRI2,Wed 28 (n=3),16:30,18:30,2 h


**Observaciones**

1. En 11 de las 12 combinaciones todo queda asignado, sin ninguna sesión de menos de 1 h y con `lab FDSI` y `Quiz ARSW` en una sola sesión. La combinación restante deja 1 h sin ubicar: es el límite conocido del paso 6 (capacidad total suficiente, pero el orden de prioridad deja un hueco sin usar) y no un problema de sesiones cortas.
2. En la tabla (prioridades `(0.9, 0.5, 0.5, 0.5, 0.5)`, perfil uniforme) hay 15 sesiones y ninguna dura menos de 1 h: `Quiz ARSW` (3 h) queda entera el martes de 17:00 a 20:00 y `lab FDSI` (2 h) entera el miércoles de 18:30 a 20:30. Los huecos de 30 min que dejan los días que empiezan a las 12:30 y 13:30 los absorben las sesiones largas de `Proyecto ARSW` (2 h 30 el martes y 3 h el miércoles).
3. El plan tiene 34 h para 34,5 h libres, así que casi no hay holgura: aun así se cumple la regla de no dejar sesiones de 30 min.


## 5.4 Prueba con planes aleatorios

Genera planes al azar (1 a 5 actividades, franjas y topes variados), se queda con los que el validador declara **factibles** y les asigna prioridades y perfiles aleatorios, es decir, sin la ayuda del MLP. En todos se ejecuta `verificar`, y se cuenta cuántos quedan **completamente asignados**.

In [13]:
rng = random.Random(7)
NIV = ["baja", "media", "alta"]
FRANJAS = [(("18:00", "21:00"),), (("08:00", "10:00"), ("14:00", "16:30")), (("19:00", "20:30"),), (("16:00", "17:50"), ("19:00", "21:00"))]


def plan_aleatorio():
    actividades = [act(f"A{i}", rng.randint(3, 14), rng.choice([1, 1.5, 2, 3, 4, 5, 6]), rng.choice(NIV), rng.choice(NIV))
                   for i in range(rng.randint(1, 5))]
    disponibilidad = [{"dia": f(n), "franjas": franjas(*rng.choice(FRANJAS))} for n in range(16) if rng.random() < 0.85]
    return plan(actividades, disponibilidad, rng.choice([60, 90, 120]), rng.choice([None, 3, 4]))


def prediccion_aleatoria():
    w = [rng.random() for _ in range(3)]
    return pr(rng.random(), [x / sum(w) for x in w])


factibles = completos = sesiones = cortas = unicas = partidas = 0
for _ in range(2000):
    d = plan_aleatorio()
    if not d["disponibilidad"]:
        continue
    p = PlanEntrada(**d)
    if validar(p, INICIO).estado == Estado.INVIABLE:
        continue
    factibles += 1
    r = asignar(p, [prediccion_aleatoria() for _ in p.actividades], INICIO)
    verificar(d, r, INICIO)
    completos += all(a.estado == EstadoAsignacion.ASIGNADA for a in r.actividades)
    sesiones += len(r.sesiones)
    cortas += sum(1 for s in r.sesiones if _minutos(s.fin) - _minutos(s.inicio) < SESION_MIN)
    for a in p.actividades:   # prioridad media o baja que cabe en una sesión: ¿quedó en una sola?
        if a.prioridad != Nivel.ALTA and max(1, int(a.horas_estimadas * 2 + 0.5)) <= p.duracion_sesion // BLOQUE_MIN:
            unicas += 1
            partidas += len([s for s in r.sesiones if s.actividad == a.nombre]) > 1

print(f"{factibles} planes factibles; todas las reglas de verificar se cumplieron en los {factibles}")
print(f"{completos} quedaron completamente asignados ({completos / factibles:.1%})")
print(f"{sesiones} sesiones en total; {cortas} duran menos de 1 h ({cortas / sesiones:.1%})")
print(f"{unicas} actividades de prioridad media o baja que caben en una sesión; {partidas} quedaron partidas ({partidas / unicas:.1%})")

1732 planes factibles; todas las reglas de verificar se cumplieron en los 1732
1669 quedaron completamente asignados (96.4%)
15200 sesiones en total; 565 duran menos de 1 h (3.7%)
991 actividades de prioridad media o baja que caben en una sesión; 4 quedaron partidas (0.4%)


**Observaciones**

1. De los 2000 planes generados (semilla 7), 1732 los acepta el validador. En los 1732 se cumplieron todas las reglas de `verificar`: **0 sesiones fuera de la disponibilidad** también con planes que ningún caso escrito a mano anticipó.
2. 1669 (96.4 %) quedaron completamente asignados. En el 3.6 % restante alguna actividad quedó `parcial` o `no_asignada` aunque el validador declaró el plan factible: es el límite conocido del paso 6 (la prioridad aleatoria deja que una actividad de plazo lejano ocupe bloques que otra de plazo cercano necesitaba). En todos esos casos la capa lo reporta en vez de forzarlo.
3. **Sesión única:** de 991 actividades de prioridad media o baja que caben en una sesión, solo 4 quedaron partidas (con la versión anterior eran 217). Las 4 son planes muy cargados (en 2 de ellos además quedan horas sin ubicar), donde no había un hueco seguido libre cuando le tocaba a la actividad; en ese caso se parte antes que dejarla sin ubicar.
4. **Sesiones cortas:** de 15200 sesiones, 565 (3.7 %) duran menos de 1 h; la versión anterior (piezas de 1 h sin las reglas nuevas) dejaba 1060 de 15640 (6.8 %). 258 de las 565 son planes con sesión máxima de 1 h y actividades de 1.5 h, 2.5 h o 3.5 h, donde sale 1 h más 30 min y no hay otra forma. Las otras 307 aparecen en planes con la disponibilidad al límite o con franjas de número impar de bloques (por ejemplo 19:00 a 20:30). Frente al reparto original en bloques de 30 min, en una prueba equivalente el 72.6 % de las sesiones duraban menos de 1 h.
5. Las horas que no se logran ubicar bajaron de 122.5 h a 118 h entre los 1732 planes y ningún plan quedó con más horas faltantes que antes (7 quedaron con menos). El día más cargado promedia 2.40 h (2.42 h con la versión anterior).


# PASO 6. MIGRACIÓN A ARCHIVOS `.py`

| Celdas del notebook | Archivo | Imports que se agregan |
|---|---|---|
| Paso 2, celda de modelos, y Paso 3, celda del algoritmo | `studyflow/asignacion.py` (una sola, en ese orden) | `from models import Actividad, DisponibilidadDia, PlanEntrada` y `from disponibilidad import BLOQUE_MIN, minutos` |
| Pasos 2.1, 3.1, 4 y 5 | `tests/` | los helpers (`act`, `dias`, `plan`, `dia`, `pr`) y `verificar` van a `conftest.py`; `CASOS` se reutiliza con `pytest.mark.parametrize` |

Cuando la lógica esté en `.py`, el notebook importa desde los módulos y deja de definirla.

Al migrar, en `disponibilidad.py` conviene renombrar `_minutos` a `minutos`, porque ahora lo usan otros módulos. No cambia nada más.

## Puntos para tener en cuenta en las otras capas

* **Conexión con el MLP:** por cada actividad, en el mismo orden que `plan.actividades`, se crea `Prediccion(prioridad=float(salida_a[i]), perfil=[float(x) for x in salida_b[i]])`. La conversión a `float` es necesaria porque Keras entrega `float32`.
* **Front y LLM redactor:** `ResultadoAsignacion.model_dump_json()` entrega todo lo necesario. El validador de salida compara el texto del LLM con `sesiones` y `actividades` (en especial las `parcial` y `no_asignada`, que el LLM debe mencionar y no ocultar).
* **Replanificación:** quien llame a `asignar` arma el `PlanEntrada` solo con lo pendiente (`horas_estimadas` = horas que faltan), pone `fecha_inicio` en el primer día replanificable y pasa en `ocupadas` las sesiones ya cumplidas o fijadas de esos días. El validador corre antes en modo `"replanificacion"`.
* **Sesión mínima y sesión única:** `SESION_MIN` (60 min) vive en `asignacion.py`. Para volver a repartir en bloques de 30 min basta con ponerla en 30. La regla de sesión única (prioridad media o baja que cabe en `duracion_sesion`) se desactiva con `reglas=False` en `_asignar`; `asignar` ya hace esa comparación en el respaldo. Con `respaldo=False` solo prueba los dos órdenes con reglas, que es lo que usa la capa de órdenes en su primer intento.
* **Límite conocido:** la asignación por prioridad no garantiza ubicar todo lo que el validador declaró factible. Si una actividad de prioridad alta y plazo lejano toma bloques que una de plazo cercano necesitaba, la segunda queda `parcial` o `no_asignada` aunque el plan completo cupiera. Siempre se reporta, nunca se oculta. La prueba 5.3 mide qué tan frecuente es con prioridades aleatorias (independientes del plazo); queda por medirla con las prioridades reales del MLP, que sí incluyen la urgencia.